<a href="https://colab.research.google.com/github/OJB-Quantum/Notebooks-for-Ideas/blob/main/IBM_Granite4_1_30b_in_Colab_96GB_VRAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deploying IBM Granite 4.1:30b in Google Colaboratory with `uv` and Ollama

Authored by Onri Jay Benally (2026)

Open Access (CC-BY-4.0)


## Primer

This notebook deploys the IBM Granite 4.1 30B language model in a Google Colaboratory GPU runtime using `uv` for Python package management and Ollama for local inference serving. The configuration targets a high memory Colab runtime with roughly 96 GB of visible GPU memory and includes checks that report the actual device inventory before the model is pulled.

Granite 4.1 is IBM's dense language model family in 3B, 8B, and 30B sizes. The 30B instruction model is appropriate for longer technical prompts, code generation, retrieval augmented generation, tool use experiments, and structured JSON output. Ollama exposes the requested runtime model as the lowercase tag `granite4.1:30b`, and this notebook preserves the human readable model label as `Granite 4.1:30b`.

| Term | Meaning |
|---|---|
| Colab | Google Colaboratory, a cloud hosted Jupyter environment. |
| GPU | Graphics Processing Unit, the accelerator used for model inference. |
| VRAM | GPU memory used for model weights, key value cache, and runtime buffers. |
| KV cache | The key value cache that stores attention state across the active context. |
| Ollama | A local model server and client interface for running open models. |
| `uv` | A fast Python package installer and resolver. |
| Context window | The maximum prompt plus generation token budget accepted by the model runtime. |

The notebook follows a linear execution path. It verifies runtime hardware, installs system dependencies, installs `uv`, installs Ollama, builds an Ollama server environment with long context settings, restarts `ollama serve`, pulls `granite4.1:30b`, warm loads the model, installs the Python client with `uv pip`, runs a smoke test, and exposes a persistent chat loop.

The high memory configuration sets `OLLAMA_CONTEXT_LENGTH=131072`, enables Flash Attention, uses `OLLAMA_KV_CACHE_TYPE=q8_0`, keeps parallel requests at one, keeps a single loaded model, and requests full GPU layer residency during every chat call. The expected validation result is `100% GPU` in `ollama ps` and a context value close to `131072` after the warm load completes.


## Control knobs

Adjust these values before running the installation and inference cells.

| Name | Meaning |
|---|---|
| `llm_version` | Human readable model label retained as `Granite 4.1:30b`. |
| `model_name` | Ollama model tag used by pull and chat calls. |
| `ollama_host` | Host interface for the local Ollama server. |
| `ollama_port` | TCP port used by the local Ollama server. |
| `models_dir` | Directory for downloaded Ollama model files. |
| `log_path` | Log file for `ollama serve`. |
| `cuda_visible_devices` | GPU selector exposed to Ollama, with `auto` detecting all visible NVIDIA devices. |
| `num_ctx` | Requested context window size for the server and every model call. |
| `num_predict` | Maximum generated tokens per response. |
| `num_batch` | Ollama prompt batch size, which can improve throughput if memory remains available. |
| `num_gpu` | Requested GPU layer residency. The large value requests full GPU placement. |
| `main_gpu` | Primary CUDA device index used by the runtime. |
| `kv_cache_type` | KV cache precision. `q8_0` lowers memory pressure, and `f16` can be tested if VRAM remains available. |
| `flash_attention` | Enables memory efficient attention required for quantized KV cache behavior. |
| `num_parallel` | Number of parallel requests. Use `1` for maximum single session context. |
| `max_loaded_models` | Number of models allowed to stay resident at the same time. |
| `keep_alive` | How long Ollama keeps the model loaded after a request. |
| `sched_spread` | Enables Ollama scheduler spreading across visible GPUs in multi GPU runtimes. |
| `temperature` | Sampling temperature used for chat calls. |
| `top_p` | Nucleus sampling threshold used for chat calls. |
| `uv_bin_dir` | Install location for the `uv` binary. |
| `install_uv` | Whether to install `uv`. |
| `install_ollama` | Whether to install Ollama. |


In [ ]:
import json
import os
import shutil
import socket
import subprocess
import time
import urllib.error
import urllib.request
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Optional


# ---------------------------------------------------------------------------
# Control knobs
# ---------------------------------------------------------------------------

TARGET_CONTEXT_TOKENS = 131_072
TARGET_KV_CACHE_TYPE = "q8_0"
TARGET_NUM_BATCH = 1024
TARGET_NUM_PARALLEL = 1
TARGET_KEEP_ALIVE = "24h"
TARGET_MAX_LOADED_MODELS = 1
FORCE_MULTI_GPU_SPREAD = True
REQUEST_FULL_GPU_LAYERS = 999


@dataclass(frozen=True)
class ColabOllamaConfig:
    """Configuration for running Granite 4.1:30b with high VRAM use."""

    llm_version: str = "Granite 4.1:30b"
    model_name: str = "granite4.1:30b"
    ollama_host: str = "127.0.0.1"
    ollama_port: int = 11434
    models_dir: Path = Path("/content/ollama_models")
    log_path: Path = Path("/content/ollama_serve.log")
    cuda_visible_devices: str = "auto"
    num_ctx: int = TARGET_CONTEXT_TOKENS
    num_predict: int = 512
    num_batch: int = TARGET_NUM_BATCH
    num_gpu: int = REQUEST_FULL_GPU_LAYERS
    main_gpu: int = 0
    temperature: float = 0.20
    top_p: float = 0.90
    kv_cache_type: str = TARGET_KV_CACHE_TYPE
    flash_attention: str = "1"
    num_parallel: int = TARGET_NUM_PARALLEL
    max_loaded_models: int = TARGET_MAX_LOADED_MODELS
    keep_alive: str = TARGET_KEEP_ALIVE
    sched_spread: bool = FORCE_MULTI_GPU_SPREAD
    uv_bin_dir: Path = Path("/content/.local/bin")
    install_uv: bool = True
    install_ollama: bool = True

    @property
    def base_url(self) -> str:
        """Return the Ollama HTTP base URL."""
        return f"http://{self.ollama_host}:{self.ollama_port}"


CFG = ColabOllamaConfig()
ENV: Optional[dict[str, str]] = None

print(json.dumps(asdict(CFG), indent=2, default=str))


## Runtime helper functions

These functions keep the remaining cells compact and provide deterministic checks around command execution, GPU visibility, TCP server readiness, Ollama server environment construction, model warm loading, and runtime diagnostics.


In [ ]:
def _is_root() -> bool:
    """Return True if the current process has root privileges."""
    try:
        return os.geteuid() == 0
    except AttributeError:
        return False


def _sudo_prefix() -> str:
    """Return a sudo prefix if root privileges are absent."""
    return "" if _is_root() else "sudo "


def run_bash(
    command: str,
    *,
    check: bool = True,
    env: Optional[dict[str, str]] = None,
) -> subprocess.CompletedProcess[str]:
    """Run a bash command in a Colab friendly way."""
    print(f"\n[run] {command}\n")
    return subprocess.run(
        ["bash", "-lc", command],
        check=check,
        env=env,
        text=True,
    )


def capture_bash(
    command: str,
    *,
    env: Optional[dict[str, str]] = None,
) -> str:
    """Run a bash command and capture stdout as text."""
    out = subprocess.check_output(
        ["bash", "-lc", command],
        env=env,
        text=True,
    )
    return out.strip()


def prepend_to_path(path: Path) -> None:
    """Prepend a directory to PATH for later subprocess calls."""
    path_str = str(path)
    path_parts = os.environ.get("PATH", "").split(":")
    if path_str not in path_parts:
        os.environ["PATH"] = f"{path_str}:{os.environ.get('PATH', '')}"


def is_tcp_port_open(host: str, port: int, timeout_s: float = 0.25) -> bool:
    """Return True if a TCP port accepts connections."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout_s)
        return sock.connect_ex((host, port)) == 0


def wait_for_ollama_ready(
    host: str,
    port: int,
    timeout_s: float = 60.0,
    poll_s: float = 0.5,
) -> dict[str, Any]:
    """Wait until the Ollama server responds to GET /api/version."""
    url = f"http://{host}:{port}/api/version"
    deadline = time.time() + timeout_s
    last_err: Optional[Exception] = None

    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=2.0) as resp:
                payload = resp.read().decode("utf-8")
            return json.loads(payload)
        except (urllib.error.URLError, json.JSONDecodeError) as err:
            last_err = err
            time.sleep(poll_s)

    raise TimeoutError(f"Ollama did not become ready: {last_err}")


def nvidia_smi_summary() -> str:
    """Return a concise GPU summary from nvidia-smi."""
    command = "command -v nvidia-smi >/dev/null 2>&1"
    if subprocess.call(["bash", "-lc", command]) != 0:
        return "nvidia-smi absent. Select a Colab GPU runtime."

    query = (
        "nvidia-smi --query-gpu=index,name,driver_version,"
        "memory.total,memory.free --format=csv,noheader,nounits"
    )
    return capture_bash(query)


def parse_gpu_memory_gib(gpu_summary: str) -> list[float]:
    """Parse total GPU memory values in GiB from nvidia-smi CSV output."""
    memory_gib: list[float] = []

    for line in gpu_summary.splitlines():
        parts = [part.strip() for part in line.split(",")]
        if len(parts) < 4:
            continue

        try:
            memory_gib.append(float(parts[3]) / 1024.0)
        except ValueError:
            continue

    return memory_gib


def warn_for_granite4_1_30b_capacity(
    gpu_summary: str,
    target_gib: float = 90.0,
) -> None:
    """Print a Granite 4.1 30B capacity note from visible GPU memory."""
    memory_gib = parse_gpu_memory_gib(gpu_summary)
    upper = gpu_summary.upper()

    if not memory_gib:
        print(
            "[gpu] WARNING: GPU memory could not be parsed. Confirm that "
            "the runtime is attached to a high memory GPU before pulling "
            "the model."
        )
        return

    total_memory = sum(memory_gib)
    max_memory = max(memory_gib)
    print(f"[gpu] Largest visible GPU memory: {max_memory:.1f} GiB")
    print(f"[gpu] Total visible GPU memory: {total_memory:.1f} GiB")

    if "G4" in upper or total_memory >= target_gib:
        print(
            "[gpu] High memory runtime detected. This is the intended "
            f"target for {CFG.llm_version} with a large context window."
        )
        return

    if max_memory >= 40.0:
        print(
            "[gpu] The model can often load with quantized weights on "
            "this class of device, although a smaller `CFG.num_ctx` may "
            "be required."
        )
        return

    print(
        "[gpu] WARNING: This runtime is likely undersized for comfortable "
        f"{CFG.llm_version} inference. Reduce `CFG.num_ctx` or use a "
        "larger GPU."
    )


def detect_cuda_devices() -> str:
    """Return all visible CUDA device indices as a comma separated string."""
    command = (
        "nvidia-smi --query-gpu=index,memory.total,name "
        "--format=csv,noheader,nounits"
    )
    result = subprocess.run(
        ["bash", "-lc", command],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if result.returncode != 0:
        return "0"

    device_indices: list[str] = []
    print("Detected NVIDIA GPUs:")
    for line in result.stdout.strip().splitlines():
        parts = [item.strip() for item in line.split(",", maxsplit=2)]
        if len(parts) != 3:
            continue

        index, memory_mib, name = parts
        device_indices.append(index)
        print(f"  GPU {index}: {name}, {int(memory_mib):,} MiB")

    if not device_indices:
        return "0"

    return ",".join(device_indices)


def build_ollama_environment(cfg: ColabOllamaConfig) -> dict[str, str]:
    """Build the environment used by the Ollama server."""
    cuda_devices = (
        detect_cuda_devices()
        if cfg.cuda_visible_devices == "auto"
        else cfg.cuda_visible_devices
    )
    cfg.models_dir.mkdir(parents=True, exist_ok=True)

    env = os.environ.copy()
    env.update(
        {
            "CUDA_VISIBLE_DEVICES": cuda_devices,
            "OLLAMA_HOST": f"{cfg.ollama_host}:{cfg.ollama_port}",
            "OLLAMA_MODELS": str(cfg.models_dir),
            "OLLAMA_CONTEXT_LENGTH": str(cfg.num_ctx),
            "OLLAMA_FLASH_ATTENTION": cfg.flash_attention,
            "OLLAMA_KV_CACHE_TYPE": cfg.kv_cache_type,
            "OLLAMA_NUM_PARALLEL": str(cfg.num_parallel),
            "OLLAMA_MAX_LOADED_MODELS": str(cfg.max_loaded_models),
            "OLLAMA_KEEP_ALIVE": cfg.keep_alive,
            "OLLAMA_LOAD_TIMEOUT": "30m",
        }
    )

    if cfg.sched_spread:
        env["OLLAMA_SCHED_SPREAD"] = "1"

    return env


def print_environment_summary(env: dict[str, str]) -> None:
    """Print the Ollama environment variables that affect VRAM use."""
    keys = [
        "CUDA_VISIBLE_DEVICES",
        "OLLAMA_MODELS",
        "OLLAMA_CONTEXT_LENGTH",
        "OLLAMA_FLASH_ATTENTION",
        "OLLAMA_KV_CACHE_TYPE",
        "OLLAMA_NUM_PARALLEL",
        "OLLAMA_MAX_LOADED_MODELS",
        "OLLAMA_KEEP_ALIVE",
        "OLLAMA_SCHED_SPREAD",
    ]
    print("\nEffective Ollama server environment:")
    for key in keys:
        print(f"  {key}={env.get(key, '')}")


def start_ollama_server(
    cfg: ColabOllamaConfig,
    env: dict[str, str],
    *,
    force_restart: bool = True,
) -> subprocess.Popen[str]:
    """Start Ollama with the supplied high VRAM environment."""
    if force_restart:
        run_bash("pkill -f 'ollama serve'", check=False)
        time.sleep(2)

    if is_tcp_port_open(cfg.ollama_host, cfg.ollama_port):
        print(
            "[ollama] Server already listening on "
            f"{cfg.ollama_host}:{cfg.ollama_port}."
        )
        return subprocess.Popen(["true"])

    cfg.log_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"[ollama] Logging to: {cfg.log_path}")
    log_file = cfg.log_path.open("a", encoding="utf-8")
    process = subprocess.Popen(
        ["ollama", "serve"],
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(f"[ollama] Started server PID={process.pid}")
    wait_for_ollama_ready(cfg.ollama_host, cfg.ollama_port)
    return process


def ollama_generate(
    cfg: ColabOllamaConfig,
    payload: dict[str, Any],
) -> dict[str, Any]:
    """Call the Ollama generate endpoint and return parsed JSON."""
    request = urllib.request.Request(
        f"{cfg.base_url}/api/generate",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=1200) as response:
        return json.loads(response.read().decode("utf-8"))


def warm_load_model(cfg: ColabOllamaConfig) -> None:
    """Warm load the model using the target long context settings."""
    payload = {
        "model": cfg.model_name,
        "prompt": "Return the single word ready.",
        "stream": False,
        "keep_alive": cfg.keep_alive,
        "options": {
            "num_ctx": cfg.num_ctx,
            "num_predict": 1,
            "num_batch": cfg.num_batch,
            "num_gpu": cfg.num_gpu,
            "main_gpu": cfg.main_gpu,
            "temperature": 0.0,
            "top_p": cfg.top_p,
        },
    }
    _ = ollama_generate(cfg, payload)


def print_ollama_gpu_diagnostics(
    env: Optional[dict[str, str]] = None,
) -> None:
    """Print Ollama and NVIDIA diagnostics for model residency."""
    run_bash("ollama ps", check=False, env=env)
    run_bash(
        "nvidia-smi --query-gpu=index,name,memory.used,memory.total,"
        "utilization.gpu --format=csv",
        check=False,
        env=env,
    )


def chat_options() -> dict[str, int | float]:
    """Return shared Ollama chat generation options."""
    return {
        "num_ctx": CFG.num_ctx,
        "num_predict": CFG.num_predict,
        "num_batch": CFG.num_batch,
        "num_gpu": CFG.num_gpu,
        "main_gpu": CFG.main_gpu,
        "temperature": CFG.temperature,
        "top_p": CFG.top_p,
    }


## Runtime sanity checks

Run this cell first. It confirms GPU visibility, reports GPU memory, and checks available disk space under `/content`, which is where the model directory is located by default.


In [ ]:
gpu_summary = nvidia_smi_summary()
print(gpu_summary)
warn_for_granite4_1_30b_capacity(gpu_summary)

run_bash("df -h /content")
run_bash("uname -a")


## Install baseline packages, `uv`, and Ollama

This cell installs Linux utilities, installs `uv` in unmanaged mode, installs Ollama using the official Linux installer, and verifies that both command line tools are available.


In [ ]:
sudo = _sudo_prefix()

if CFG.install_ollama or CFG.install_uv:
    run_bash(f"{sudo}apt-get update -y")
    run_bash(
        f"{sudo}apt-get install -y "
        "curl ca-certificates zstd pciutils lshw"
    )

if CFG.install_uv:
    CFG.uv_bin_dir.mkdir(parents=True, exist_ok=True)
    run_bash(
        "curl -LsSf https://astral.sh/uv/install.sh | "
        f'env UV_UNMANAGED_INSTALL="{CFG.uv_bin_dir}" sh'
    )

prepend_to_path(CFG.uv_bin_dir)
run_bash("command -v uv")
run_bash("uv --version")

if CFG.install_ollama:
    run_bash("curl -fsSL https://ollama.com/install.sh | sh")

run_bash("command -v ollama")
run_bash("ollama -v")


## Start `ollama serve` and pull `granite4.1:30b`

This cell builds the high VRAM Ollama environment, restarts the local server so the environment variables take effect, pulls the requested model, warm loads the model with the target context settings, and prints the model residency diagnostics. The server log is written to `CFG.log_path`.


In [ ]:
CFG.models_dir.mkdir(parents=True, exist_ok=True)
ENV = build_ollama_environment(CFG)
print_environment_summary(ENV)

OLLAMA_PROCESS = start_ollama_server(CFG, ENV, force_restart=True)

version_payload = wait_for_ollama_ready(CFG.ollama_host, CFG.ollama_port)
print("[ollama] /api/version =>", version_payload)

run_bash(f"ollama pull {CFG.model_name}", env=ENV)
warm_load_model(CFG)

run_bash("ollama list", env=ENV)
run_bash(f"ollama show {CFG.model_name}", check=False, env=ENV)
print_ollama_gpu_diagnostics(ENV)


## Install the Ollama Python client with `uv`

This cell installs the Python client into the active Colab environment so the remaining cells can call the local server from Python.


In [ ]:
run_bash("uv pip install --system --upgrade ollama")


## Smoke test

The smoke test sends a short technical prompt to `Granite 4.1:30b`, prints the response, then reports active Ollama model placement and GPU memory utilization. The validation target is `100% GPU` in `ollama ps` with the requested long context visible after the warm load.


In [ ]:
from ollama import chat

smoke_messages = [
    {
        "role": "system",
        "content": (
            "You are a precise technical assistant. Reply in one compact "
            "paragraph unless the user requests a longer structure."
        ),
    },
    {
        "role": "user",
        "content": (
            "Confirm that you are running as Granite 4.1:30b and explain, "
            "in one paragraph, what this local Colab deployment is doing."
        ),
    },
]

response = chat(
    model=CFG.model_name,
    messages=smoke_messages,
    options=chat_options(),
    keep_alive=CFG.keep_alive,
)

print(response.message.content)
print_ollama_gpu_diagnostics(ENV)


## Streaming response test

This cell verifies token streaming, which is useful for longer technical prompts where an immediate partial response is preferable. The same long context and GPU residency options are passed into the streaming call.


In [ ]:
from ollama import chat

stream_messages = [
    {
        "role": "user",
        "content": (
            "Give a concise three point checklist for validating a local "
            "Granite 4.1:30b deployment in Colab."
        ),
    }
]

for chunk in chat(
    model=CFG.model_name,
    messages=stream_messages,
    options=chat_options(),
    keep_alive=CFG.keep_alive,
    stream=True,
):
    print(chunk.message.content, end="", flush=True)

print()


## Interactive chat loop

Run this cell after the model is installed. Type `quit`, `exit`, or `q` to stop the loop. Reduce `TARGET_CONTEXT_TOKENS` or `TARGET_NUM_BATCH` in the configuration cell if memory pressure appears during long conversations, then rerun the configuration, helper, and server cells so the new settings reach `ollama serve`.


In [ ]:
from ollama import chat

MODEL_NAME = CFG.model_name
SYSTEM_PROMPT = (
    "You are Granite 4.1:30b running locally in Google Colab. "
    "You provide concise, technically precise answers."
)

messages: list[dict[str, str]] = []
if SYSTEM_PROMPT.strip():
    messages.append({"role": "system", "content": SYSTEM_PROMPT.strip()})


def send_turn(user_text: str) -> str:
    """Send one user turn and return one assistant turn."""
    messages.append({"role": "user", "content": user_text})
    response = chat(
        model=MODEL_NAME,
        messages=messages,
        options=chat_options(),
        keep_alive=CFG.keep_alive,
    )
    assistant_text = response.message.content
    messages.append({"role": "assistant", "content": assistant_text})
    return assistant_text


while True:
    user_text = input("\nYou: ").strip()
    if user_text.lower() in {"quit", "exit", "q"}:
        print("Stopped.")
        break
    if not user_text:
        continue

    try:
        reply = send_turn(user_text)
    except Exception as exc:
        raise RuntimeError(
            "Prompting failed. Verify these earlier cells:\n"
            "  1) `ollama serve` is running on localhost:11434\n"
            f"  2) the model is present with `ollama pull {CFG.model_name}`\n"
            "  3) the `ollama` Python package is installed\n"
            "  4) `CFG.num_ctx` fits the visible GPU memory\n"
            "  5) `ollama ps` reports GPU residency\n"
        ) from exc

    print(f"\nAssistant: {reply}")


## Save a transcript

Run this optional cell after the interactive loop to save the conversation into `/content/granite_4_1_30b_chat_transcript.json`.


In [ ]:
transcript_path = Path("/content/granite_4_1_30b_chat_transcript.json")

if "messages" not in globals() or not messages:
    print("No chat transcript is available yet.")
else:
    transcript_path.write_text(
        json.dumps(messages, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print(f"Saved transcript to {transcript_path}")


## Operational notes

Reconnecting to a new Colab runtime means the local server, packages, and downloaded model may need to be recreated. The default model directory is `/content/ollama_models`, so the model files disappear when the runtime is recycled unless they are copied to persistent storage. The default model tag in this notebook is `granite4.1:30b`, matching the requested `Granite 4.1:30b` version label.

The high VRAM path depends on the server receiving its environment before `ollama serve` starts. Rerun the server cell after changing `TARGET_CONTEXT_TOKENS`, `TARGET_KV_CACHE_TYPE`, `TARGET_NUM_BATCH`, `TARGET_NUM_PARALLEL`, or the CUDA device selector. The expected diagnostic target is `100% GPU` in `ollama ps`. If any CPU fraction appears, keep `TARGET_KV_CACHE_TYPE="q8_0"`, reduce `TARGET_NUM_BATCH` to `512`, restart the server cell, and inspect the diagnostics again. If the model remains fully resident with several GB of free VRAM, `TARGET_KV_CACHE_TYPE="f16"` can be tested for a higher precision KV cache.

Parallel requests multiply memory pressure by the number of parallel sessions. The notebook keeps `TARGET_NUM_PARALLEL=1` so the largest practical single conversation context receives the full memory budget.
